In [0]:
%pip install simplemma

In [0]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql import types as T

BRONZE_TABLE = "workspace.linguacopilot.bronze_events"
SILVER_TABLE = "workspace.linguacopilot.silver_events"

# Events created manually while testing the Lambda and the API
TEST_EVENT_IDS = [
    "d250beda-e8b2-4a33-a21d-f94353877557",
    "341ff007-e45f-4183-9cae-770cf24f715a",
    "86f0d40b-b0a0-4619-b8c3-280abe774068",
]


@F.pandas_udf(T.StringType())
def lemmatize_de(words: pd.Series) -> pd.Series:
    import simplemma
    return words.map(lambda w: simplemma.lemmatize(w, lang="de") if w else None)


silver = (
    spark.table(BRONZE_TABLE)
    .filter(~F.col("event_id").isin(TEST_EVENT_IDS))
    .filter(F.col("word").isNotNull())
    # Protects against the same file being ingested twice
    .dropDuplicates(["event_id"])
    # A double-click on the same word at the same moment counts as one click
    .dropDuplicates(["video_id", "timestamp_sec", "word"])
    .select(
        "event_id",
        F.trim("word").alias("word"),
        lemmatize_de(F.trim("word")).alias("lemma"),
        F.trim("sentence").alias("sentence"),
        "word_translation",
        "sentence_translation",
        F.coalesce("source_lang", F.lit("DE")).alias("source_lang"),
        F.coalesce("target_lang", F.lit("FR")).alias("target_lang"),
        "video_id",
        F.col("timestamp_sec").cast("double").alias("timestamp_sec"),
        F.to_timestamp("captured_at").alias("captured_at"),
        F.to_timestamp("ingested_at").alias("ingested_at"),
        "_source_file",
    )
)

silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(SILVER_TABLE)
display(spark.table(SILVER_TABLE))